# Library Importation

In [ ]:
"""
Import the nesseceray libraries use for this workflow
"""

import pandas as pd               # Data manipulation and analysis
import numpy as np                # Numerical operations
import matplotlib.pyplot as plt   # Core plotting library
import seaborn as sns             # Statistical data visualization built on matplotlib

from statsmodels.tsa.holtwinters import ExponentialSmoothing, Holt
    # For time series forecasting using the Holt–Winters exponential smoothing method

from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
    # Performance metrics to evaluate forecasting accuracy

# Use seaborn's default color palette for all plots
color_pal = sns.color_palette()

from datetime import datetime     # For handling date and time information

# Supress warnings for cleaner output
import warnings
warnings.filterwarnings("ignore")

# Generate a timestamp (formatted as "YYYY-MM-DD_HH-MM-SS") representing the current moment
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# Data Importation, Cleaning and Transformation

### Ingest the raw dataset and select relevant columns: ['Date', 'Total Sales']

In [ ]:
# Load raw dataset
df = pd.read_excel("dataset.xlsx")

# Select relevant columns safely
df = df[['Date', 'Total Sales']].copy()

df = df.reset_index(drop=True) # Reset index to ensure a clean DataFrame


##### Convert the Date column to a proper datetime format so Python can interpret it correctly, then sort the dataset by date, remove invalid entries, and scale the Total Sales values to billions for easier computation.

In [ ]:
# Convert Date column
df['Date'] = pd.to_datetime(df['Date'], format='%b-%y', errors='coerce')

# Drop invalid dates or missing rows
df = df.dropna(subset=['Date', 'Total Sales'])

# Sort chronologically
df = df.sort_values('Date')

# Clean Total Sales values
df['Total Sales'] = (
    df['Total Sales']
    .astype(str)
    .str.replace('₦', '')
    .str.replace(',', '')
    .str.strip()
    .astype(float)
)

# Convert sales to billions
df['Sales_Billions'] = df['Total Sales'] / 1e9

# Set index for time series operations
df = df.set_index('Date')


print('Some summary information about the dataset:')
print(df.info())
df.head()


# EDA - Exploratory Data Analysis
###### in this section, performed some analysis to better understand the data

#### Plot a line chart of Total Sales to visualize how sales have evolved over the years and identify key trends.

In [ ]:
plt.figure(figsize=(16, 6))
plt.plot(df.index, df['Sales_Billions'], linewidth=2)

plt.title("Total Sales Over Time")
plt.ylabel("Sales (₦ Billions)")
plt.xlabel("Date")

plt.grid(True, alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
print("====== Dataset Overview ========")
print(f"• Reporting Period: {df.index.min().strftime('%B %Y')} to {df.index.max().strftime('%B %Y')}")
print(f"• Total Number of Months: {len(df)}")
print(f"• Sales Range: ₦{df['Total Sales'].min():,.2f}  —  ₦{df['Total Sales'].max():,.2f}")
print("================================")

### create a reusable function to be use to make chart with different variables

In [ ]:
# Plot
def plot(df, col):
    plt.figure(figsize=(16, 6))
    plt.plot(df.index, df[col], linewidth=2)
    plt.title(f"{col} Over Time")
    plt.ylabel(col)
    plt.xlabel("Date")
    plt.grid(True, alpha=0.5)
    plt.tight_layout()
    plt.show()

## Calculate MoM Growth Rate

In [ ]:
# Month-over-Month Growth Rate
df["MoM_Pct_%"] = (df["Total Sales"].pct_change() * 100).round(2)
plot(df, "MoM_Pct_%")


## MoM Absolute Growth

In [ ]:
df["MoM_Abs"] = df["Total Sales"].diff()
# Plot MoM Absolute Growth
plot(df, "MoM_Abs")


The Month-over-Month Absolute Change (MoM_Abs) shows initially high fluctuations in the first few months of 2021, indicating uneven early-stage sales increases. However, from late 2021 onward, MoM_Abs remains almost constant (≈ 476M), reflecting a steady linear upward trend where sales increase by nearly the same amount every month.

This indicates stable monthly growth with no significant seasonal or irregular variations.

# calculate Year-over-Year Growth Rate`

In [ ]:
# Year-over-Year Growth Rate`
df["YoY_Growth_%"] = (df["Total Sales"].pct_change(12) * 100).round(2)
# Plot YoY Growth Rate
plot(df, "YoY_Growth_%")


The YoY growth rate starts extremely high, exceeding 200% in early 2022 because sales increased sharply from a very low base in 2021. Over time, the YoY growth rate declines steadily, dropping to about 22–25% by late 2025.

This downward trajectory is expected for a series that grows by a constant absolute amount each month. As the sales base becomes larger, the same absolute increase represents a smaller percentage change.

The smooth decay pattern also confirms the absence of volatility or seasonality in the data, consistent with the linear upward trend observed in the time series.

# Split Train/Test

In [ ]:
test_size = 12  # last 12 months for testing

train = df.iloc[:-test_size]
test = df.iloc[-test_size:]
print(f"Training data from {train.index.min()} to {train.index.max()}")
print(f"Testing data from {test.index.min()} to {test.index.max()}")

In [ ]:
# Plot the split
fig, ax = plt.subplots(figsize=(15, 5))

ax.plot(train.index, train['Sales_Billions'], label='Training Set', linewidth=2)
ax.plot(test.index, test['Sales_Billions'], label='Test Set', linewidth=2)

# Add split marker automatically
split_point = test.index.min()
ax.axvline(split_point, color='black', linestyle='--', linewidth=1)


# Titles and labels
ax.set_title('Train–Test Split of Total Sales (Billions)')
ax.set_ylabel('Sales (₦ Billions)')
ax.set_xlabel('Date')

ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()


# Holt’s Linear Trend Model

In [ ]:
# Build and fit Holt's Linear Trend model

# Holt model
model = Holt(train['Total Sales'],
                  exponential=False,  # additive trend
                  damped_trend=False  # optional: set True if you want trend damping
                 ).fit(
                     optimized=True
                 )

# Forecasting

In [ ]:
# Forecast
timeframe = 12 # months
test_pred = model.forecast(timeframe)
test_pred.index = test.index  # Align forecast with test index

# Combine for comparison
results = pd.DataFrame({
    "Actual": test['Total Sales'],
    "Predicted": test_pred.round(1)
})

# Compute difference
results['Difference'] = results['Actual'] - results['Predicted'] # subtract predicted from actual

results['Difference'] = results['Difference'].round(1) # round the value to one decimal place

results.head()


## Model Evaluation Metrics 

In [ ]:

# MAE: Mean Absolute Error
# Measures the average size of the errors between actual and predicted values.
# Lower MAE indicates more accurate predictions.
mae = mean_absolute_error(results['Actual'], results['Predicted'])


# RMSE: Root Mean Squared Error
# Penalizes large errors more heavily than MAE.
# Useful when large deviations are especially undesirable.
rmse = np.sqrt(mean_squared_error(results['Actual'], results['Predicted']))


# MAPE: Mean Absolute Percentage Error (in %)
# Shows prediction accuracy as a percentage.
# Interpreted as: "On average, the model's predictions are X% off from actual values."
mape = (abs(results['Actual'] - results['Predicted']) / results['Actual']).mean() * 100


print(f"\nError Metrics:")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.2f}%")


In [ ]:
# Forecast Plot (Styled like the Train–Test Split)

fig, ax = plt.subplots(figsize=(15, 5))

# Plot training data
ax.plot(train.index, train['Sales_Billions'], 
        label='Training Data', linewidth=2)

# Plot actual test data
ax.plot(test.index, test['Sales_Billions'], 
        label='Actual Test Data', linewidth=2)

# Plot forecast
ax.plot(test_pred.index, test_pred / 1e9, 
        label='Forecast (Holt)', linewidth=2)

# Vertical line marking start of test set
split_point = test.index.min()
ax.axvline(split_point, color='black', linestyle='--', linewidth=1)

# Titles & labels
ax.set_title("Holt Forecast – Test Period")
ax.set_ylabel("Sales (₦ Billions)")
ax.set_xlabel("Date")

ax.grid(True, alpha=0.3) # add a grid for better readability
ax.legend()

plt.tight_layout() # ensure everything fits without overlap
plt.show() # display the plot


# Hyperparameter tuning: grid search over α and β

In [ ]:
best_mape = float('inf')
best_alpha = None
best_beta = None

alphas = np.arange(0.1, 1.0, 0.1)
betas = np.arange(0.1, 1.0, 0.1)

for alpha in alphas:
    for beta in betas:

        model = None

        # Try new parameter name
        try:
            model = Holt(train['Total Sales'], damped_trend=False).fit(
                smoothing_level=alpha,
                smoothing_trend=beta,
                optimized=False
            )
        except:
            pass

        # Try legacy parameter name
        if model is None:
            try:
                model = Holt(train['Total Sales'], damped_trend=False).fit(
                    smoothing_level=alpha,
                    smoothing_slope=beta,
                    optimized=False
                )
            except:
                continue

        # Forecast
        pred = pd.Series(model.forecast(len(test)), index=test.index)

        # Calculate MAPE
        mape = mean_absolute_percentage_error(test['Total Sales'], pred)

        # Track best
        if mape < best_mape:
            best_mape = mape
            best_alpha = alpha
            best_beta = beta

# print the best parameter
print("===== Best Parameter =====")
print(f"Best α: {best_alpha}") 
print(f"Best β: {best_beta}")
print(f"Best MAPE: {best_mape:.4f}")
print("==========================")


In [ ]:
# Build and fit Holt's Linear Trend model

# Holt model
model = Holt(train['Total Sales'],
                  exponential=False,  # additive trend
                  damped_trend=False  # optional: set True if you want trend damping
                 ).fit(
                     smoothing_level=alpha,    # α: model responsiveness to new data
                     smoothing_slope=beta,      # β: how quickly trend updates
                     optimized=False            # use manual values
                 )

"""
Parameter meanings:

smoothing_level (α):
    Controls how fast the model reacts to new data.
    - High α → reacts quickly, more sensitive
    - Low α  → reacts slowly, smoother forecasts
    α = 0.34 → model gives 34% weight to latest value.

smoothing_trend (β):
    Controls how fast the trend (slope) updates.
    - High β → trend changes rapidly
    - Low β  → trend remains stable
    β = 0.01 → trend updates very slowly.
"""

In [ ]:
# forecast on the train data
test_pred = model.forecast(test_size)

# Combine for comparison
results = pd.DataFrame({
    "Actual": test['Total Sales'],
    "Predicted": test_pred.round(1)
})

# Compute difference
results['Difference'] = results['Actual'] - results['Predicted']

results['Difference'] = results['Difference'].round(1)

results.head()

In [ ]:
# Calculate error metrics
mae = mean_absolute_error(results['Actual'], results['Predicted'])
rmse = np.sqrt(mean_squared_error(results['Actual'], results['Predicted']))
mape = (abs(results['Actual'] - results['Predicted']) / results['Actual']).mean() * 100

print(f"\nError Metrics:")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.2f}%")



In [ ]:
# Forecast Plot (Styled like the Train–Test Split)

fig, ax = plt.subplots(figsize=(15, 5))

# Plot training data
ax.plot(train.index, train['Sales_Billions'], 
        label='Training Data', linewidth=2)

# Plot actual test data
ax.plot(test.index, test['Sales_Billions'], 
        label='Actual Test Data', linewidth=2)

# Plot forecast
ax.plot(test_pred.index, test_pred / 1e9, 
        label='Forecast (Holt)', linewidth=2)

# Vertical line marking start of test set
split_point = test.index.min()
ax.axvline(split_point, color='black', linestyle='--', linewidth=1)

# Titles & labels
ax.set_title("Holt Forecast – Test Period")
ax.set_ylabel("Sales (₦ Billions)")
ax.set_xlabel("Date")

ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()


# Run the model with on full dataset

In [ ]:
# Holt model
model = Holt(df['Total Sales'],
                  exponential=False,  # additive trend
                  damped_trend=False  # optional: set True if you want trend damping
                 ).fit(
                     smoothing_level=alpha,    # α: model responsiveness to new data
                     smoothing_slope=beta,      # β: how quickly trend updates
                     optimized=False            # use manual values
                 )

# Forecast Into the Future (Business Use Case)

In [ ]:
# Forecast for the next 12 months
future_months = 12
# Generate forecast for the next 12 months
future_forecast = model.forecast(future_months)

In [ ]:
"""
get the last date in the dataset to determin when are forecast starting from,
and setting the time range for forecasting to 1 years
"""
future_df = pd.DataFrame({
    "Date": pd.date_range(
        start=df.index[-1] + pd.offsets.MonthBegin(),
        periods=future_months,
        freq='MS'
    ),
    "Forecast": future_forecast
}).set_index("Date")

In [ ]:
# combine historical and forecast data for reporting
report_data = pd.concat([df, future_df], axis=0)
report_data.to_csv(f"forecast_data_{timestamp}.csv", index_label="Date")

# save the model for reuse

In [ ]:
# Build file name
filename = f"holt_model_{timestamp}.pkl"

# Save model
model.save(filename)

print(f"Model saved as: {filename}")


In [ ]:
plt.figure(figsize=(15,5))

plt.plot(df.index, df["Sales_Billions"], label="Actual Sales", linewidth=3)
plt.plot(future_df.index, future_df["Forecast"] / 1e9,
         label="Forecast ", linestyle="--", linewidth=3)

plt.axvline(df.index.max(), color='black', linestyle='--', linewidth=1)

plt.title("12-Month Sales Forecast")
plt.ylabel("Sales (₦ Billions)")
plt.xlabel("Date")

plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
# Save the figure — before plt.show()
plt.savefig(f"forecast_chart_{timestamp}.png", dpi=300, bbox_inches="tight")
plt.show() # display the chart
